In [0]:
import json
import requests

from datetime import datetime, timezone

In [0]:
BASE_URL = "https://api.worldbank.org/v2"

START_YEAR = 2000
END_YEAR = 2024

print(BASE_URL)
print(START_YEAR)
print(END_YEAR)

In [0]:
country_code = "BRA"
indicator_code = "NY.GDP.MKTP.CD"

print(country_code)
print(indicator_code)

In [0]:
request_url = (
    f"{BASE_URL}/country/{country_code}"
    f"/indicator/{indicator_code}"
)

request_params = {
    "format": "json",
    "date": f"{START_YEAR}:{END_YEAR}",
    "per_page": 100
}

print(request_url)
print(request_params)

In [0]:
response = requests.get(
    request_url,
    params=request_params,
    timeout=30
)

response.raise_for_status()

print(response.status_code)
print(response.url)

In [0]:
payload = response.json()

print(type(payload))
print(len(payload))

In [0]:
metadata = payload[0]
observations = payload[1]

print(metadata)
print(len(observations))

In [0]:
ingestion_timestamp = datetime.now(timezone.utc)

print(ingestion_timestamp)

In [0]:
ingestion_id = ingestion_timestamp.strftime("%Y%m%dT%H%M%S%fZ")

print(ingestion_id)

In [0]:
raw_payload = json.dumps(
    payload,
    ensure_ascii=False
)

print(type(raw_payload))
print(len(raw_payload))

In [0]:
bronze_record = {
    "ingestion_id": ingestion_id,
    "ingestion_timestamp": ingestion_timestamp,
    "country_code": country_code,
    "indicator_code": indicator_code,
    "request_url": response.url,
    "http_status_code": response.status_code,
    "raw_payload": raw_payload
}

print(bronze_record.keys())

In [0]:
df_bronze = spark.createDataFrame([bronze_record])

df_bronze.printSchema()

In [0]:
display(
    df_bronze.select(
        "ingestion_id",
        "ingestion_timestamp",
        "country_code",
        "indicator_code",
        "http_status_code",
        "request_url"
    )
)

In [0]:
bronze_record_count = df_bronze.count()

print(bronze_record_count)

In [0]:
assert bronze_record_count == 1, "Quantidade inesperada de registros Bronze"
assert bronze_record["http_status_code"] == 200, "A requisição não foi bem-sucedida"
assert bronze_record["raw_payload"], "O payload bruto está vazio"

print("Registro Bronze validado com sucesso!")

In [0]:
from pyspark.sql import functions as F

display(
    df_bronze.select(
        "ingestion_id",
        F.length("raw_payload").alias("raw_payload_length")
    )
)

In [0]:
print(df_bronze.columns)

In [0]:
df_bronze_organizado = df_bronze.select(
    "ingestion_id",
    "ingestion_timestamp",
    "country_code",
    "indicator_code",
    "http_status_code",
    "request_url",
    "raw_payload"
)

print(df_bronze_organizado.columns)

In [0]:
colunas_esperadas = [
    "ingestion_id",
    "ingestion_timestamp",
    "country_code",
    "indicator_code",
    "http_status_code",
    "request_url",
    "raw_payload"
]

assert df_bronze_organizado.columns == colunas_esperadas, (
    "A ordem das colunas Bronze está incorreta"
)

print("Ordem das colunas Bronze validada com sucesso!")

In [0]:
bronze_table_name = "world_bank_bronze"

print(bronze_table_name)

In [0]:
current_catalog = spark.sql(
    "SELECT current_catalog() AS catalog_name"
).first()["catalog_name"]

print(current_catalog)

In [0]:
current_schema = spark.sql(
    "SELECT current_schema() AS schema_name"
).first()["schema_name"]

print(current_schema)

In [0]:
spark.sql("""
    CREATE CATALOG IF NOT EXISTS global_economic_analytics
    COMMENT 'Dados e ativos do projeto de análise macroeconômica global'
""")

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS global_economic_analytics.bronze
    COMMENT 'Dados brutos ingeridos das fontes externas'
""")

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS global_economic_analytics.silver
    COMMENT 'Dados limpos, tipados, deduplicados e validados'
""")

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS global_economic_analytics.gold
    COMMENT 'Dados agregados e preparados para análises macroeconômicas'
""")

In [0]:
display(
    spark.sql("""
        SHOW SCHEMAS IN global_economic_analytics
    """)
)

In [0]:
bronze_full_table_name = (
    "global_economic_analytics.bronze.world_bank_raw"
)

print(bronze_full_table_name)

In [0]:
table_exists = spark.catalog.tableExists(bronze_full_table_name)

print(table_exists)

In [0]:
(
    df_bronze_organizado.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_full_table_name)
)

print("Tabela Bronze gravada com sucesso!")

In [0]:
table_exists_after_write = spark.catalog.tableExists(
    bronze_full_table_name
)

print(table_exists_after_write)

In [0]:
display(
    spark.table(bronze_full_table_name).select(
        "ingestion_id",
        "ingestion_timestamp",
        "country_code",
        "indicator_code",
        "http_status_code"
    )
)

In [0]:
bronze_table_count = spark.table(
    bronze_full_table_name
).count()

print(bronze_table_count)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE DETAIL {bronze_full_table_name}
    """).select(
        "format",
        "name",
        "location",
        "numFiles",
        "sizeInBytes"
    )
)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE HISTORY {bronze_full_table_name}
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    )
)

In [0]:
df_bronze_version_0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .table(bronze_full_table_name)
)

print(df_bronze_version_0.count())

In [0]:
display(
    df_bronze_version_0.select(
        "ingestion_id",
        "country_code",
        "indicator_code",
        "http_status_code"
    )
)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE TABLE EXTENDED {bronze_full_table_name}
    """)
)

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {bronze_full_table_name}
    IS 'Payloads brutos das requisições realizadas à API do Banco Mundial.'
""")

print("Comentário da tabela adicionado com sucesso!")

In [0]:
table_comment = spark.sql(f"""
    SELECT comment
    FROM global_economic_analytics.information_schema.tables
    WHERE table_schema = 'bronze'
      AND table_name = 'world_bank_raw'
""").first()["comment"]

print(table_comment)

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    ALTER COLUMN ingestion_id
    COMMENT 'Identificador único do lote de ingestão.'
""")

print("Coluna ingestion_id documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    ALTER COLUMN ingestion_timestamp
    COMMENT 'Data e hora da ingestão do lote em UTC.'
""")

print("Coluna ingestion_timestamp documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    ALTER COLUMN country_code
    COMMENT 'Código ISO 3 do país consultado na API do Banco Mundial.'
""")

print("Coluna country_code documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    ALTER COLUMN indicator_code
    COMMENT 'Código do indicador consultado na API do Banco Mundial.'
""")

print("Coluna indicator_code documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    ALTER COLUMN http_status_code
    COMMENT 'Código de status HTTP retornado pela API.'
""")

print("Coluna http_status_code documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    ALTER COLUMN request_url
    COMMENT 'URL completa utilizada na requisição à API.'
""")

print("Coluna request_url documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    ALTER COLUMN raw_payload
    COMMENT 'Resposta completa da API preservada em formato JSON.'
""")

print("Coluna raw_payload documentada com sucesso!")

In [0]:
display(
    spark.sql(f"""
        DESCRIBE TABLE {bronze_full_table_name}
    """)
)

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    SET TBLPROPERTIES (
        'project' = 'global_economic_analytics',
        'data_source' = 'World Bank API',
        'data_layer' = 'bronze',
        'data_granularity' = 'one row per API request'
    )
""")

print("Propriedades da tabela Bronze adicionadas com sucesso!")

In [0]:
table_properties = spark.sql(f"""
    SHOW TBLPROPERTIES {bronze_full_table_name}
""")

display(
    table_properties.filter(
        """
        key IN (
            'project',
            'data_source',
            'data_layer',
            'data_granularity'
        )
        """
    )
)

In [0]:
existing_batch_count = (
    spark.table(bronze_full_table_name)
    .filter(F.col("ingestion_id") == ingestion_id)
    .count()
)

print(existing_batch_count)

In [0]:
if existing_batch_count == 0:
    print("Lote disponível para gravação.")
else:
    print("Lote já existente. Gravação ignorada.")

In [0]:
def batch_already_exists(table_name, batch_id):
    return (
        spark.table(table_name)
        .filter(F.col("ingestion_id") == batch_id)
        .limit(1)
        .count()
        > 0
    )

In [0]:
batch_exists = batch_already_exists(
    bronze_full_table_name,
    ingestion_id
)

print(batch_exists)

In [0]:
def write_bronze_batch(df, table_name, batch_id):
    if batch_already_exists(table_name, batch_id):
        print(f"Lote {batch_id} já existe. Gravação ignorada.")
        return False

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )

    print(f"Lote {batch_id} gravado com sucesso.")
    return True

In [0]:
write_result = write_bronze_batch(
    df_bronze_organizado,
    bronze_full_table_name,
    ingestion_id
)

print(write_result)

In [0]:
display(
    spark.table(bronze_full_table_name)
    .groupBy(
        "ingestion_id",
        "country_code",
        "indicator_code"
    )
    .count()
    .orderBy("ingestion_id")
)


In [0]:
current_bronze_count = spark.table(
    bronze_full_table_name
).count()

print(current_bronze_count)

In [0]:
request_key = (
    f"{country_code}|"
    f"{indicator_code}|"
    f"{START_YEAR}|"
    f"{END_YEAR}"
)

print(request_key)

In [0]:
import hashlib

request_key_hash = hashlib.sha256(
    request_key.encode("utf-8")
).hexdigest()

print(request_key_hash)

In [0]:
request_key_hash_short = request_key_hash[:16]

print(request_key_hash_short)

In [0]:
payload_hash = hashlib.sha256(
    raw_payload.encode("utf-8")
).hexdigest()

print(payload_hash)

In [0]:
idempotency_key = hashlib.sha256(
    f"{request_key_hash}|{payload_hash}".encode("utf-8")
).hexdigest()

print(idempotency_key)

In [0]:
bronze_record_v2 = {
    **bronze_record,
    "request_key_hash": request_key_hash,
    "payload_hash": payload_hash,
    "idempotency_key": idempotency_key
}

print(bronze_record_v2.keys())

In [0]:
df_bronze_v2 = spark.createDataFrame([bronze_record_v2])

df_bronze_v2.printSchema()

In [0]:
df_bronze_v2_organizado = df_bronze_v2.select(
    "ingestion_id",
    "ingestion_timestamp",
    "idempotency_key",
    "request_key_hash",
    "payload_hash",
    "country_code",
    "indicator_code",
    "http_status_code",
    "request_url",
    "raw_payload"
)

print(df_bronze_v2_organizado.columns)

In [0]:
colunas_bronze_v2_esperadas = [
    "ingestion_id",
    "ingestion_timestamp",
    "idempotency_key",
    "request_key_hash",
    "payload_hash",
    "country_code",
    "indicator_code",
    "http_status_code",
    "request_url",
    "raw_payload"
]

assert (
    df_bronze_v2_organizado.columns
    == colunas_bronze_v2_esperadas
), "A ordem das colunas Bronze V2 está incorreta"

print("Ordem das colunas Bronze V2 validada com sucesso!")

In [0]:
current_bronze_columns = spark.table(
    bronze_full_table_name
).columns

print(current_bronze_columns)

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    ADD COLUMNS (
        idempotency_key STRING
        COMMENT 'Chave determinística da requisição combinada com o conteúdo retornado.'
    )
""")

print("Coluna idempotency_key adicionada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    ADD COLUMNS (
        request_key_hash STRING
        COMMENT 'Hash SHA-256 que identifica país, indicador e período da requisição.'
    )
""")

print("Coluna request_key_hash adicionada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {bronze_full_table_name}
    ADD COLUMNS (
        payload_hash STRING
        COMMENT 'Hash SHA-256 do conteúdo bruto retornado pela API.'
    )
""")

print("Coluna payload_hash adicionada com sucesso!")

In [0]:
display(
    spark.sql(f"""
        DESCRIBE TABLE {bronze_full_table_name}
    """)
)

In [0]:
display(
    spark.table(bronze_full_table_name)
    .select(
        "ingestion_id",
        "idempotency_key",
        "request_key_hash",
        "payload_hash"
    )
    .orderBy("ingestion_id")
)

In [0]:
spark.sql(f"""
    UPDATE {bronze_full_table_name}
    SET
        idempotency_key = '{idempotency_key}',
        request_key_hash = '{request_key_hash}',
        payload_hash = '{payload_hash}'
    WHERE country_code = '{country_code}'
      AND indicator_code = '{indicator_code}'
      AND idempotency_key IS NULL
""")

print("Chaves históricas preenchidas com sucesso!")

In [0]:
display(
    spark.table(bronze_full_table_name)
    .select(
        "ingestion_id",
        "idempotency_key",
        "request_key_hash",
        "payload_hash"
    )
    .orderBy("ingestion_id")
)

In [0]:
display(
    spark.table(bronze_full_table_name)
    .groupBy("idempotency_key")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
from pyspark.sql.window import Window

deduplication_window = (
    Window
    .partitionBy("idempotency_key")
    .orderBy(
        F.col("ingestion_timestamp").asc(),
        F.col("ingestion_id").asc()
    )
)

df_bronze_ranked = (
    spark.table(bronze_full_table_name)
    .withColumn(
        "duplicate_rank",
        F.row_number().over(deduplication_window)
    )
)

display(
    df_bronze_ranked.select(
        "ingestion_id",
        "ingestion_timestamp",
        "idempotency_key",
        "duplicate_rank"
    ).orderBy("duplicate_rank")
)

In [0]:
df_bronze_deduplicated = (
    df_bronze_ranked
    .filter(F.col("duplicate_rank") == 1)
    .drop("duplicate_rank")
)

print(df_bronze_deduplicated.count())

In [0]:
display(
    df_bronze_deduplicated.select(
        "ingestion_id",
        "ingestion_timestamp",
        "idempotency_key"
    )
)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE HISTORY {bronze_full_table_name}
    """).select(
        "version",
        "timestamp",
        "operation"
    ).orderBy(F.col("version").desc())
)

In [0]:
duplicate_ingestion_id = "20261101T233624039748Z"

display(
    spark.table(bronze_full_table_name)
    .filter(F.col("ingestion_id") == duplicate_ingestion_id)
    .select(
        "ingestion_id",
        "ingestion_timestamp",
        "country_code",
        "indicator_code",
        "idempotency_key"
    )
)

In [0]:
spark.sql(f"""
    DELETE FROM {bronze_full_table_name}
    WHERE ingestion_id = '{duplicate_ingestion_id}'
""")

print("Registro duplicado removido com sucesso!")

In [0]:
bronze_count_after_delete = spark.table(
    bronze_full_table_name
).count()

print(bronze_count_after_delete)

In [0]:
display(
    spark.table(bronze_full_table_name).select(
        "ingestion_id",
        "ingestion_timestamp",
        "country_code",
        "indicator_code",
        "idempotency_key"
    )
)

In [0]:
def idempotency_key_already_exists(table_name, key):
    return (
        spark.table(table_name)
        .filter(F.col("idempotency_key") == key)
        .limit(1)
        .count()
        > 0
    )

In [0]:
idempotency_key_exists = idempotency_key_already_exists(
    bronze_full_table_name,
    idempotency_key
)

print(idempotency_key_exists)

In [0]:
def write_bronze_batch_v2(df, table_name, key):
    if idempotency_key_already_exists(table_name, key):
        print(
            f"Conteúdo com a chave {key} já existe. "
            "Gravação ignorada."
        )
        return False

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )

    print(f"Conteúdo com a chave {key} gravado com sucesso.")
    return True

In [0]:
write_result_v2 = write_bronze_batch_v2(
    df_bronze_v2_organizado,
    bronze_full_table_name,
    idempotency_key
)

print(write_result_v2)

In [0]:
final_bronze_count = spark.table(
    bronze_full_table_name
).count()

print(final_bronze_count)